# ZTE Parallax — three vantage points on the same minds

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victor-iyi/zte/blob/feature/decoder/notebooks/zte_parallax.ipynb)

**The confound, and the structural answer.** ZuCo's task is fully confounded with its stimulus set: Cramér's
$V(\text{task}, \text{stimulus}) = 0.998$, and no sentence appears under two tasks. A model trained across tasks can
therefore win its contrastive game on task register rather than content — and the measured encoder does exactly
that, *amplifying* the task probe to 0.918 from a raw-feature 0.685. Parallax removes the confound structurally
instead of adversarially: **three independent encoders, one per task (NR, SR, TSR)**, each trained only on its own
task's readings with the current best-measured recipe (residual coding off, gallery CE off — both falsified by
their matched ablations on 2026-08-15). Within one task there is no task variable left to lean on.

**The prize.** The scientific readout is the **3 × 3 cross-task transfer matrix**. A model trained on task X and
evaluated on task Y ≠ X faces a never-seen subject (LOSO holdout `ZAB`) reading stimuli disjoint from everything the
model was trained on — the strongest generalisation cell this project can produce. The honest expectation: the
in-task diagonal well above chance, the cross-task cells possibly null. **A null is a finding** and is reported
plainly. The metaphor is parallax — three vantage points observing the same minds; what stays fixed across vantage
points, measured by transfer and by CKA between the three models, is the task-invariant thought code.

**The readout is closed-set.** Stratified retrieval rank percentile with a bootstrap CI, plus the menu-capacity
audit over exact-length pools — within one task the task dimension is constant by construction, and exact-length
pools remove ZuCo's free 5.14 bits of sentence length. **Free generation is not a parallax deliverable**: the
encoder's bit budget prices it out, and no generation number appears in this study.

| You want to | Go to |
| --- | --- |
| Set up a fresh Colab runtime | §1 – §3 |
| Train the three vantage points | §4 |
| Fill the 3 × 3 transfer matrix | §5 |
| Build the report and walk the chamber | §6 |
| Read the outcome honestly | §7 |

## 1 · Provision the runtime

Colab ships an older Python than ZTE requires (`>=3.14`), so `uv` provisions the pinned interpreter and installs
everything into a cached virtualenv. The clone is shallow and hard-resets to the branch, so only git-tracked files
move — your Drive folder and any local cache are untouched. Re-running this cell on a warm runtime is fast and
idempotent.

In [ ]:
%%bash
pip install -q uv
# Work whether this is a fresh runtime (/content), a re-run already inside zte/, or a restored session.
if [ -f pyproject.toml ]; then :
elif [ -d zte/.git ]; then cd zte
else git clone --depth 1 https://github.com/victor-iyi/zte.git --branch feature/decoder && cd zte
fi
git fetch --depth 1 origin feature/decoder && git reset --hard FETCH_HEAD
echo "ZTE @ $(git rev-parse --short HEAD): $(git log -1 --pretty=%s)"
uv python install 3.14
uv sync --all-groups

## 2 · Wire the kernel

Colab's kernel is an **older interpreter than the 3.14 ZTE requires**, so `import zte` here is a `SyntaxError`. It
never needs to: every capability arrives through `zte-colab`, one subcommand per question, each printing a single
JSON object on stdout with its logs on stderr. `colab()` below is that bridge and the only way this notebook
reaches the package — everything else in a code cell is the standard library, `IPython.display`, `google.colab`,
and Colab's own renderers.

In [ ]:
import json
import os
import platform
import subprocess
from typing import Any


def colab(command: str, *args: str) -> dict[str, Any]:
    """Runs one `zte-colab` subcommand in the provisioned venv and returns the JSON object it printed.

    This is the notebook's only route into ZTE. The package runs on 3.14 inside the uv venv; this kernel is
    Colab's own older interpreter, so it renders payloads rather than computing them.
    """
    argv = ['uv', 'run', 'zte-colab', command, *args]
    done = subprocess.run(argv, capture_output=True, text=True, check=False)
    if done.returncode != 0:
        raise RuntimeError(f'`{" ".join(argv)}` failed:\n{done.stderr[-3000:]}')

    return json.loads(done.stdout)


# Enter the repo in the notebook kernel, so relative paths and every subprocess resolve. A %%bash `cd` cannot
# do this: it dies with its own shell.
if os.path.isdir('zte') and not os.path.isfile('pyproject.toml'):
    os.chdir('zte')

ENV = colab('env')
os.environ.update(ENV['env'])

try:
    from google.colab import userdata  # type: ignore[import-untyped]

    _hf = userdata.get('HF_TOKEN')
except Exception as exc:  # not on Colab, or the secret is not granted to this notebook
    _hf, _ = None, print(f'HF_TOKEN unavailable ({type(exc).__name__}) — Hub downloads will be unauthenticated.')
if _hf:
    os.environ['HF_TOKEN'] = _hf
    print('HF_TOKEN loaded — authenticated HuggingFace Hub downloads enabled.')

print(f'repo   : {ENV["root"]}')
print(f'venv   : Python {ENV["venv"]["python"]} · zte {ENV["venv"]["zte"]}   ← every `!uv run` command')
print(f'kernel : Python {platform.python_version()}   ← this cell; renders payloads, never imports zte')

## 3 · Drive is the workspace

A Colab VM can vanish without warning, so everything durable lives on Drive: one shared folder for data plus one
dated folder per session.

```text
Sharables/ZTE/
├── prepared/       # cached feature bundles, NOT date-stamped: built once, reused forever
├── ZuCo Dataset/   # the raw .mat archives — task1 (SR), task2 (NR), task3 (TSR)
└── YYYY-MM-DD/     # one folder per session: experiments/, analysis/, archives/
```

Training checkpoints go to the VM's fast local disk and are mirrored to Drive after every run, because a Drive FUSE
stall mid-`torch.save` is a torn checkpoint. Everything else — the transfer cells' aggregate report, the chamber —
is written straight to Drive. To resume an interrupted session, set `RESUME_DATE` to that session's folder name;
every `--resume` then finds its work already done.

In [ ]:
from google.colab import drive  # type: ignore[import-untyped]

drive.mount('/gdrive')

In [ ]:
# Set to an existing folder name (e.g. '2026-08-15') to resume that session; None starts today's.
RESUME_DATE: str | None = None
# 'local+mirror' trains on the VM disk and copies to Drive after each run (recommended).
WRITE_MODE: str = 'local+mirror'
ZTE_DRIVE: str = '/gdrive/My Drive/Sharables/ZTE'

_resume = ('--resume-date', RESUME_DATE) if RESUME_DATE else ()
SESSION = colab('session', '--drive', ZTE_DRIVE, '--write-mode', WRITE_MODE, *_resume)

# Every `!` command below inherits these, so the bundle cache, the data root and the backup target are wired once.
os.environ.update(SESSION['env'])

RUN_DATE: str = SESSION['run_date']
DATA_DIR: str = SESSION['data_dir']
DRIVE_ANALYSIS: str = SESSION['drive_analysis']
OUT_ROOT: str = SESSION['out_root']
DRIVE_BACKUP: str = SESSION['drive_backup']
PREPARED_LOCAL: str = SESSION['prepared_local']
PREPARED_DRIVE: str = SESSION['prepared_drive']

print(f'session   : {RUN_DATE}   ({"resumed" if SESSION["resumed"] else "new"})')
print(f'Drive     : {SESSION["drive_root"]}   (mounted: {SESSION["drive_mounted"]})')
print(f'raw data  : {DATA_DIR}   (present: {SESSION["data_dir_present"]})')
print(f'runs ->   : {OUT_ROOT}   (backed up to {DRIVE_BACKUP})')
print(f'analysis  : {DRIVE_ANALYSIS}')
print(f'prepared  : {PREPARED_DRIVE}   (staged on the VM at {PREPARED_LOCAL})')

### 3a · Helpers this notebook uses everywhere

Small and boring: how runs move between the VM and Drive in both directions, plus the resource readout. Each is a
thin renderer over a `zte-colab` payload, so the exclusion rules live in the package and are tested there rather
than drifting in a notebook.

In [ ]:
def _mirror(direction: str, date: str, sub: str, local: str | None) -> None:
    """Runs one mirror and reports what moved, or why nothing did."""
    where = ['--drive', ZTE_DRIVE, '--write-mode', WRITE_MODE, *(('--local', local) if local else ())]
    payload = colab('mirror', *where, '--direction', direction, '--date', date, '--sub', sub)

    if reason := payload['skipped_reason']:
        print(f'nothing mirrored: {reason}')
        return

    print(f'{payload["src"]} -> {payload["dst"]}   ({payload["copied"]} copied, {payload["failed"]} failed)')


def mirror_to_drive(local: str | None = None, sub: str = 'experiments') -> None:
    """Copy the VM's runs to Drive, minus what is rebuildable, so the session survives the machine."""
    _mirror('up', RUN_DATE, sub, local)


def restore_from_drive(run_date: str | None = None, sub: str = 'experiments', local: str | None = None) -> None:
    """Pull a session's runs back to the VM so every `--resume` finds its work after a runtime reset."""
    _mirror('down', run_date or RUN_DATE, sub, local)


def show_resources() -> None:
    """Prints RAM / GPU / disk as they stand, so an out-of-memory kill is predictable rather than a mystery."""
    res = colab('env')['resources']
    gpu = f'{res["gpu"]["name"]} ({res["gpu"]["total_gb"]} GB)' if res['gpu'] else 'none'
    print(f'RAM {res["ram_gb"]} GB · {res["cpu_count"]} cores · {res["free_disk_gb"]} GB free disk · GPU {gpu}')


show_resources()

### 3b · The data — prepare once, and which vantage points are present

`zte-prepare` keys each parallax config by a hash of the fields that actually change the processed bundle and
builds only what the persistent Drive store does not already hold, so a fully-prepared project never touches the
raw `.mat` files again.

**TSR needs the task3 archives** in the Drive dataset folder; ZuCo ships them separately. If they are absent the
cell says so and the study proceeds as a 2 × 2 with NR and SR — every loop below reads `TASKS`, so nothing else
needs editing. Drop the archives in and re-run this cell to widen the matrix to 3 × 3.

In [ ]:
import pathlib

CONFIGS: dict[str, str] = {
    'NR': 'experiments/parallax/parallax_nr.yaml',
    'SR': 'experiments/parallax/parallax_sr.yaml',
    'TSR': 'experiments/parallax/parallax_tsr.yaml',
}

_root = pathlib.Path(DATA_DIR)
_tsr = list(_root.rglob('results*_TSR.mat')) or list(_root.rglob('*task3*'))
TASKS: list[str] = ['NR', 'SR'] + (['TSR'] if _tsr else [])

if 'TSR' not in TASKS:
    print('TSR absent: no task3 archives under the dataset folder, so that vantage point is skipped.')
    print('Add the task3 (TSR) archives to the Drive dataset folder and re-run this cell to include it.')
print('vantage points:', ', '.join(TASKS))

CONFIG_ARGS: str = ' '.join(f'"{CONFIGS[task]}"' for task in TASKS)

!uv run zte-prepare --root "{DATA_DIR}" --configs {CONFIG_ARGS} \
  --cache-dir "{PREPARED_LOCAL}" --cache-remote "{PREPARED_DRIVE}"

## 4 · The training matrix — this is the multi-hour cell

Three vantage points × three seeds = up to nine runs of the current best-measured recipe (`exp17_base` with a
single-task `dataset.tasks`), each a few hours on a raw-conformer arm. Every run is **resumable**: a finished run
is skipped, an interrupted one continues from its last checkpoint, and each run mirrors to Drive as it completes,
so a reclaimed VM costs at most one epoch. Start it, come back, and re-run the cell verbatim after any
interruption.

A single seed is not a result on this corpus — arms whose only difference was noise have moved between 2 and 9
hits in 700. Three seeds are the floor for an error bar; the comment in the cell names the 5-seed variant.

In [ ]:
HOLDOUT: str = 'ZAB'
# Three seeds are the floor for an error bar; extend to (42, 43, 44, 45, 46) for the 5-seed variant.
SEEDS: tuple[int, ...] = (42, 43, 44)

for task in TASKS:
    for seed in SEEDS:
        name = f'parallax_{task.lower()}_lo{HOLDOUT}_s{seed}'
        print(f'\n=== {name} ' + '=' * 44)
        !uv run zte-run \
          --config "{CONFIGS[task]}" --root "{DATA_DIR}" --name "{name}" --out-root "{OUT_ROOT}" \
          --loso-holdout "{HOLDOUT}" --seed {seed} --data-cache "{PREPARED_LOCAL}" \
          --drive-backup "{DRIVE_BACKUP}" --spatial exact --resume
        mirror_to_drive()

## 5 · The transfer matrix — 3 × 3 × seeds

`zte-parallax transfer` embeds the held-out subject's readings of one task's stimuli with one trained model and
scores closed-set retrieval against that task's gallery: stratified rank percentile with a bootstrap CI, the
length-matched variant beside it, and the menu-capacity audit. Post-processing is fitted on the non-holdout
subjects of the eval task, never on the holdout, and `postprocess_fit` travels in the artifact.

The off-diagonal cells are the point: a model trained on task X, evaluated on task Y, meets a subject it never saw
reading sentences disjoint from everything it was trained on. Each cell writes `transfer.json` (plus the
embeddings) into its own directory, and the loop skips a cell whose `transfer.json` already exists, so this
resumes exactly like training.

In [ ]:
TRANSFER_ROOT: str = f'{OUT_ROOT}/parallax/transfer'

for train_task in TASKS:
    for eval_task in TASKS:
        for seed in SEEDS:
            ckpt = f'{OUT_ROOT}/parallax_{train_task.lower()}_lo{HOLDOUT}_s{seed}/checkpoints/best.pt'
            cell = f'{TRANSFER_ROOT}/{train_task}_to_{eval_task}_s{seed}'
            if pathlib.Path(cell, 'transfer.json').is_file():
                print(f'done, skipping: {train_task} -> {eval_task} · s{seed}')
                continue
            if not pathlib.Path(ckpt).is_file():
                print(f'no checkpoint yet, skipping: {ckpt}')
                continue
            print(f'\n=== {train_task} -> {eval_task} · s{seed} ' + '=' * 36)
            !uv run zte-parallax transfer \
              --ckpt "{ckpt}" --eval-task {eval_task} --root "{DATA_DIR}" --out "{TRANSFER_ROOT}" \
              --holdout "{HOLDOUT}" --seed {seed}

mirror_to_drive()

## 6 · The report and the chamber

`zte-parallax report` aggregates every transfer cell into `PARALLAX.json` (the numbers, per seed), `PARALLAX.md`
(the honest prose reading), and `CHAMBER_DATA.json` (the geometry: each eval task's sentence prototypes reduced to
three dimensions and Procrustes-aligned across the three models' views, so the same sentence can be watched from
all three vantage points).

`zte-parallax chamber` renders that JSON into one self-contained page — no server, no network. The chamber is an
inspection tool, not an audit: the numbers that carry the argument are in `PARALLAX.md`, and the page exists so a
surprising number can be chased into the geometry that produced it.

In [ ]:
PARALLAX_OUT: str = f'{DRIVE_ANALYSIS}/parallax'

!uv run zte-parallax report --transfers "{TRANSFER_ROOT}" --out "{PARALLAX_OUT}"

summary = pathlib.Path(PARALLAX_OUT) / 'PARALLAX.md'
if summary.is_file():
    print(summary.read_text()[:2000])

In [ ]:
import shutil

from IPython.display import IFrame, display

CHAMBER: str = f'{PARALLAX_OUT}/CHAMBER.html'

!uv run zte-parallax chamber --report-dir "{PARALLAX_OUT}" --out "{CHAMBER}"

# Copy to the VM disk before embedding: an iframe reading straight off the Drive mount is slow enough to look broken.
local_chamber = pathlib.Path('res/analysis/CHAMBER.html')
local_chamber.parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(CHAMBER, local_chamber)
print(f'{local_chamber}  ({local_chamber.stat().st_size / 1e6:.1f} MB)  ·  also on Drive at {CHAMBER}')
display(IFrame(src=str(local_chamber), width='100%', height=820))

## 7 · How to read the results honestly

Read `PARALLAX.md` in this order, and only in this order.

1. **The diagonal first.** A vantage point evaluated on its own task, on the held-out subject: does a single-task
   encoder reach a stranger's brain at all? A diagonal at chance makes the off-diagonal uninterpretable.
2. **Then the off-diagonal.** Rank percentile with its bootstrap CI, chance at 0.5 — the never-seen subject ×
   never-seen stimuli cell, the strongest generalisation statement this project can make. A CI that includes 0.5
   is a null, and a null is a finding: report it plainly as *no measurable task-invariant transfer at this
   recipe*, not as a failure to be hidden.
3. **Then menu capacity.** The largest exact-length closed set served at the target accuracy — the honest "how
   many sentences could a menu offer" number, with the 5.14-bit length subsidy removed by construction.
4. **Then CKA between the three models.** Convergent geometry without transfer means the vantage points agree
   about structure in a way retrieval cannot yet use; transfer without convergent geometry would be the suspicious
   pattern. Either way, say which one was observed.

What is *not* here: generation (the bit budget prices it out, and no generation number appears in this study), and
pooled `sentence_retrieval` (every number above is the held-out cell). The full method, the falsifiable
predictions and the pre-registered reading of every outcome live in `docs/PARALLAX.md`.